# Instrumental variables (Balke–Pearl)

Classic binary IV: $Z \to X \to Y$ with latent confounding $X \leftrightarrow Y$.


In [ ]:
from partialid import Assumptions, CausalDiagram, Query, Variables, bound
from partialid.problems import binary_iv

# Observational law sampled from a valid IV SCM (instrumental inequalities hold)
p = binary_iv(seed=2)
print(p.name, "—", p.notes)
print("query:", p.query.describe())


: 

## Classic IV diagram (no compliance monotonicity)


In [ ]:
r = bound(p.variables, p.data, p.query, Assumptions.iv("Z", "X", "Y"))
print(r.summary(p.query))
print(f"tight={r.tight}  method={r.method}")
assert r.success and r.tight


## Optional: no-defiers (`monotone=True`)

Compliance monotonicity is an *extra* assumption. If it conflicts with $P$,
the LP is infeasible (`success=False`).


In [ ]:
r_m = bound(
    p.variables,
    p.data,
    p.query,
    Assumptions.iv("Z", "X", "Y", monotone=True),
)
print(f"monotone success={r_m.success}")
if r_m.success:
    print(r_m.summary(p.query))
else:
    print(f"infeasible: {r_m.message}")


## Build the same IV assumptions from a diagram

`Assumptions.iv` is a thin recipe over `CausalDiagram` + exclusions.


In [ ]:
g = CausalDiagram.from_edges(
    nodes=["Z", "X", "Y"],
    directed=[("Z", "X"), ("X", "Y")],
    bidirected=[("X", "Y")],
)
a = Assumptions.from_diagram(g)
r_diag = bound(p.variables, p.data, p.query, a)
print(r_diag.summary(p.query))
print("widths match recipe:", abs(r_diag.width - r.width) < 1e-6)

# Manual Variables / Query (same shape as the catalog problem)
V = Variables.from_binary("Z", "X", "Y")
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
assert V.names == p.variables.names
_ = Q  # same as p.query
